In [87]:
import os
from datetime import datetime, timedelta
from Bio import Entrez
import pandas as pd
import time
from datetime import datetime, timedelta

In [112]:
# Define the function to search PubMed with a date range
def search_pubmed_by_date(query, email, start_date, end_date, retmax=100):
    """
    Search PubMed for articles matching the query within a specific date range.
    
    Parameters:
    - query: PubMed search query
    - email: Email address for NCBI API
    - start_date: Start date in format 'YYYY/MM/DD'
    - end_date: End date in format 'YYYY/MM/DD'
    - retmax: Maximum number of results to return
    
    Returns:
    - List of PubMed IDs
    """
    Entrez.email = email
    
    # Format the date range for PubMed query
    date_range = f"{start_date}:{end_date}[Date - Create]"
    full_query = f"({query}) AND {date_range}"
    
    # Search PubMed
    handle = Entrez.esearch(db="pubmed", term=full_query, retmax=retmax)
    record = Entrez.read(handle)
    handle.close()
    
    return record["IdList"]

# Function to fetch article details
def fetch_article_details(id_list):
    """
    Fetch details for a list of PubMed IDs.
    
    Parameters:
    - id_list: List of PubMed IDs
    
    Returns:
    - List of dictionaries with article details
    """
    articles = []
    
    # Process in batches to avoid overloading the server
    batch_size = 50
    for i in range(0, len(id_list), batch_size):
        batch_ids = id_list[i:i+batch_size]
        
        try:
            handle = Entrez.efetch(db="pubmed", id=batch_ids, rettype="medline", retmode="xml")
            records = Entrez.read(handle)

            for record in records['PubmedArticle']+records['PubmedBookArticle']:
                # Extract relevant information
                article = {}
                
                if 'ArticleTitle' in record['MedlineCitation']['Article']:
                    article['title'] = record['MedlineCitation']['Article']['ArticleTitle']
                if 'Abstract' in record['MedlineCitation']['Article']:
                    article['abstract'] = record['MedlineCitation']['Article']['Abstract']['AbstractText'][0]
                    
                if len(record['MedlineCitation']['Article']['ArticleDate'])>0:
                    date=[r for r in record['MedlineCitation']['Article']['ArticleDate'] if r.attributes['DateType']=="Electronic"][0]
                elif 'DateCompleted' in record['MedlineCitation']:
                    date=record['MedlineCitation']['DateCompleted']
                else:
                    date=None
                if date is not None:
                    article['date']=f"{date['Year']}-{date['Month']}-{date['Day']}"
                    
                if 'ELocationID' in record['MedlineCitation']['Article']:
                    dois=[str(r) for r in record['MedlineCitation']['Article']['ELocationID'] if r.attributes['EIdType']=='doi']
                    if len(dois)>0:
                        article['doi']=dois[0]
                        
                articles.append(article)
            
            handle.close()
            
            # Be nice to NCBI servers
            time.sleep(0.5)
            
        except Exception as e:
            print(f"Error fetching details for batch: {e}")
    
    return articles

def list_dates(start_date, end_date):
    """
    List all dates between the start and end date, inclusive.
    
    Parameters:
    - start_date: Start date in the format 'YYYY/MM/DD'
    - end_date: End date in the format 'YYYY/MM/DD'
    
    Returns:
    - List of dates as strings in 'YYYY/MM/DD' format
    """
    # Convert string dates to datetime objects
    start = datetime.strptime(start_date, '%Y/%m/%d')
    end = datetime.strptime(end_date, '%Y/%m/%d')
    
    # Create a list of dates
    date_list = []
    current_date = start
    
    while current_date <= end:
        date_list.append(current_date.strftime('%Y/%m/%d'))
        current_date += timedelta(days=1)  # Increment by one day
    
    return date_list


# Main function to run the PubMed search for specified days
def run_pubmed_search(query, email, start_date, end_date):
    """
    Run PubMed searches for specific days and save results.
    
    Parameters:
    - query: PubMed search query
    - email: Email address for NCBI API
    """
    
    all_results = []

    days_list=list_dates(start_date, end_date)
    
    for day in days_list:
        print(f"Searching for articles on {day}...")
        
        # For a single day, use the same date for start and end
        pmids = search_pubmed_by_date(query, email, day, day)
        
        if pmids:
            print(f"Found {len(pmids)} articles for {day}")
            articles = fetch_article_details(pmids)
            
            # Add date to each article record
            for article in articles:
                article['search_date'] = day.replace('/','-')
                all_results.append(article)
            
        else:
            print(f"No articles found for {day}")

    return pd.DataFrame(all_results)

In [113]:
# Your PubMed query
query = """("protein*"[Title] OR "peptide*"[Title] OR "antibod*"[Title] OR "nanobod*"[Title] OR "enzyme*"[Title] OR "TCR"[Title]) 
AND ("learn*"[Title] OR "deep*"[Title] OR "neural network"[Title] OR "diffusion*"[Title] OR "transformer*"[Title] OR "flow matching"[Title] 
OR "predict*"[Title] OR "generative"[Title] OR "embedding"[Title] OR "representation"[Title] OR "benchmark*"[Title] OR "supervised*"[Title] 
OR "unsupervised*"[Title] OR "design*"[Title] OR "structure*"[Title] OR "model*"[Title] OR "reinforcement*"[Title])"""

# Set your email (required by NCBI)
email = "karin.hrovatin@merckgroup.com"  # Replace with your actual email
start_date, end_date = "2025/05/01","2025/05/04"

In [114]:
run_pubmed_search(query, email, start_date, end_date)

Searching for articles on 2025/05/01...
Found 12 articles for 2025/05/01
Searching for articles on 2025/05/02...
Found 18 articles for 2025/05/02
Searching for articles on 2025/05/03...
Found 9 articles for 2025/05/03
Searching for articles on 2025/05/04...
No articles found for 2025/05/04


,title,abstract,date,doi,search_date
0,Identification of non-synonymous SNPs affectin...,The genes NBN and MLH1 are critical for DNA re...,2025-05-02,10.1007/s13353-025-00968-2,2025-05-01
1,An intranasal subunit vaccine induces protecti...,Although vaccines are usually given intramuscu...,2025-05-01,10.1038/s41467-025-59353-6,2025-05-01
2,Molecular structure and protein function of mi...,Alzheimer's disease is a neurodegenerative dis...,2025-04-29,10.1016/j.ijbiomac.2025.143696,2025-05-01
3,Design of Recombinant Bacteriocin Fusion Prote...,Gastrointestinal cancer is the fifth most comm...,2025-04-29,10.1016/j.micpath.2025.107633,2025-05-01
4,Design of potent and proteolytically stable do...,The successful treatment of type 2 diabetes an...,2025-04-26,10.1016/j.bmc.2025.118215,2025-05-01
5,Formation of tiger nut protein fibrils through...,Tiger nut protein fibrils (TNPFs) were prepare...,2025-04-28,10.1016/j.foodchem.2025.144557,2025-05-01
6,Dual-Site Targeting by Peptide Inhibitors of t...,Heat shock protein 90 (Hsp90) is a pivotal mol...,2025-05-01,10.1021/acs.jcim.5c00629,2025-05-01
7,Predicted risk genotypes to cardiovascular dis...,The human genetic variability mainly involves ...,2025-05-01,10.1007/s11033-025-10537-9,2025-05-01
8,A responsive living material prepared by diffu...,Stimuli-responsive engineered living materials...,2025-05-01,10.1073/pnas.2424405122,2025-05-01
9,EVOLVE: A Web Platform for AI-Based Protein Mu...,While predicting structure-function relationsh...,2025-05-01,10.1021/acs.jcim.5c00026,2025-05-01
